# Tutorial Interactivo de DeepEval
Este notebook te guiará a través de las pruebas unitarias para LLMs usando DeepEval y modelos de Google Generative AI.

In [1]:
!pip install -q deepeval google-generativeai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 1. Configuración del Modelo de Evaluación
Configuraremos a Gemini como nuestro evaluador. DeepEval permite usar modelos personalizados definiendo una clase que herede de `DeepEvalBaseLLM`.

In [2]:
import os

import google.generativeai as genai
from deepeval.models.base_model import DeepEvalBaseLLM

# Configurar la API Key desde variables de entorno (.env)
GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

class GeminiModel(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-3.1-flash-lite"):
        self.model = genai.GenerativeModel(model_name)

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = chat_model.generate_content(prompt)
        return res.text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "gemini-3.1-flash-lite"

evaluator_model = GeminiModel()
print("Modelo evaluador (Gemini Lite) configurado correctamente.")

/tmp/ipykernel_77/3132216401.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Modelo evaluador (Gemini Lite) configurado correctamente.


## 2. Nuestra Primera Métrica: Faithfulness
La métrica de Fidelidad mide si el output del LLM se deriva puramente del contexto proporcionado. Es fundamental para evitar alucinaciones en sistemas RAG.

In [3]:
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# 1. Definimos el caso de prueba
input_text = "¿Cuál es la política de devoluciones?"
context = ["Nuestra política permite devoluciones en un plazo de 30 días, siempre que el producto esté en su empaque original."]
actual_output = "Puedes devolver tus productos en un plazo de 30 días si conservas el empaque original."

test_case = LLMTestCase(
    input=input_text,
    actual_output=actual_output,
    retrieval_context=context
)

# 2. Configuramos la métrica usando nuestro evaluador Gemini
metric = FaithfulnessMetric(
    threshold=0.7,
    model=evaluator_model,
    include_reason=True
)

# 3. Ejecutamos la evaluación
metric.measure(test_case)

print(f"Puntaje de Fidelidad: {metric.score}")
print(f"Razón del puntaje: {metric.reason}")

Output()

Puntaje de Fidelidad: 1.0
Razón del puntaje: The score is 1.00 because the actual output is perfectly aligned with the retrieval context. Everything looks great, keep up the fantastic work!


## 3. Métrica: Answer Relevancy
Esta métrica evalúa si la respuesta del modelo es concisa y aborda directamente la intención del usuario.

In [4]:
from deepeval.metrics import AnswerRelevancyMetric

# Reutilizamos el test_case anterior
relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=evaluator_model,
    include_reason=True
)

relevancy_metric.measure(test_case)

print(f"Puntaje de Relevancia: {relevancy_metric.score}")
print(f"Razón: {relevancy_metric.reason}")

Output()

Puntaje de Relevancia: 1.0
Razón: The score is 1.00 because the response provided a perfectly relevant and accurate explanation of the return policy, addressing the user's inquiry directly and concisely. Great job!


## 4. Evaluaciones en Lote (Batch Testing)
En un entorno real, no probarás una sola respuesta. DeepEval permite definir un conjunto de datos y evaluarlos todos de una vez.

In [5]:
from deepeval import evaluate

# Definimos varios casos de prueba
case_1 = LLMTestCase(
    input="¿Tienen envíos internacionales?",
    actual_output="Sí, enviamos a todo el mundo.",
    retrieval_context=["Realizamos envíos nacionales e internacionales (excepto a zonas de conflicto)."]
)

case_2 = LLMTestCase(
    input="¿Cuál es el precio del iPhone 15?",
    actual_output="El iPhone 15 cuesta 999 dólares.",
    retrieval_context=["Los precios de los smartphones varían según la región, consulta el catálogo local."]
)

# Ejecutamos una evaluación masiva
evaluate([case_1, case_2], metrics=[relevancy_metric, metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-3.1-flash-lite, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-3.1-flash-lite, strict=False, 
async_mode=True)...

Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x75312ac4dec0> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x75312ac4dec0> is already entered

...

Task was destroyed but it is pending!
task: <Task pending name='Task-1492' coro=<Kernel.shell_main() running at /usr/local/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            ¿Tienen envíos internacionales?                                                        │
│  │     Actual Output:    Sí, enviamos a todo el mundo.                                                          │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy │ 1.00  │ 0.70      │ The score is 1.00 because your response was per...        │
│        FAIL  │ Faithfulness     │ 0.00  │ 0.70      │ The score is 0.00 because the actual output incorrectly   │
│              │                  │       │           │ claims worldwide shipping, directly contradicting the     │
│              │                  │       │           │ context which specifies that shipments are not made to    │
│              │                  │       │           │ conflict zones.                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=2 | failed=0                 │ 2         │
│  Faithfulness               │ 0.50                  │ 50.00% | passed=1 | failed=1                  │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10615985;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 113.3s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because your response was perfectly focused and directly addressed the inquiry with complete accuracy. Great job providing exactly what the user needed!', strict_mode=False, flaky=False, evaluation_model='gemini-3.1-flash-lite', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Statements:\n[\n    "Sí, enviamos a todo el mundo."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Faithfulness', threshold=0.7, success=False, score=0.0, reason='The score is 0.00 because the actual output incorrectly claims worldwide shipping, directly contradicting the context which specifies that shipments are not made to conflict zones.', strict_mode=False, flaky=False, evaluation_model='gemini-3.1-flash-lite', error=Non

## 5. Métrica Personalizada: G-Eval
G-Eval es una métrica basada en LLM que permite definir criterios subjetivos o específicos de negocio mediante lenguaje natural.

In [6]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

# Definimos una métrica de 'Tono Profesional'
tone_metric = GEval(
    name="Tono Profesional",
    criteria="Determina si el output es profesional, educado y no utiliza jerga informal.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=evaluator_model,
    threshold=0.7
)

custom_test_case = LLMTestCase(
    input="¿Cómo va mi pedido?",
    actual_output="¡Qué onda! Tu pedido ya va en camino, tranqui."
)

# Intentamos medir (recuerda esperar si el error de cuota persiste)
try:
    tone_metric.measure(custom_test_case)
    print(f"Puntaje de Tono: {tone_metric.score}")
    print(f"Razón: {tone_metric.reason}")
except Exception as e:
    print(f"Error: {e}. Por favor espera un momento antes de reintentar.")

Output()

Puntaje de Tono: 0.0
Razón: The output fails to meet professional standards, using highly colloquial language ('¡Qué onda!', 'tranqui') and an overly casual tone that is inappropriate for a formal corporate environment.


## Conclusión
¡Felicidades! Has aprendido a:
1. Configurar un evaluador personalizado (Gemini).
2. Medir la Fidelidad (Faithfulness) para evitar alucinaciones.
3. Medir la Relevancia de las respuestas.
4. Ejecutar pruebas en lote (Batch).
5. Crear métricas personalizadas con G-Eval.